### Step 1: Setup and Installation

In [11]:
!pip install langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv -q

In [12]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### Step 2: Connect to MCP Server

In [13]:
import sys
import asyncio

# Work around Jupyter's stderr fileno() issue
if sys.platform == "win32":
    import mcp.client.stdio

    original_stdio_client = mcp.client.stdio.stdio_client

    from functools import partial

    mcp.client.stdio.stdio_client = partial(
        original_stdio_client,
        errlog=open(os.devnull, "w")
    )

print("Jupyter MCP compatibility workaround applied.")


Jupyter MCP compatibility workaround applied.


In [67]:
from pathlib import Path
from langchain_mcp_adapters.client import MultiServerMCPClient

# Use an absolute path / create this directory first if it does not exist.
DOCUMENTS_DIR = Path("./mcp_documents").resolve()
DOCUMENTS_DIR.mkdir(exist_ok=True)

print(f"MCP server will be allowed to access MCP folder")

mcp_client = MultiServerMCPClient(
    {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-filesystem",
                DOCUMENTS_DIR,
            ],
        }
    }
)

print("MCP client configured.")

MCP server will be allowed to access MCP folder
MCP client configured.


In [18]:
print("Available MCP tools:")
for index, tool in enumerate(tools, start=1):
    print(f"\nTool {index}")
    print(f"Name: {tool.name}")
    print(f"Description: {tool.description}")

Available MCP tools:

Tool 1
Name: read_file
Description: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.

Tool 2
Name: read_text_file
Description: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.

Tool 3
Name: read_media_file
Description: Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.

Tool 4
Name: read_multiple_files
Description: Read the contents of multiple f

### Step 3:  Load MCP Tools into LangChain

In [22]:
mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(mcp_tools)} tools from MCP server(s)")
print("\nAvailable tools:")
for index, tool in enumerate(mcp_tools, start=1):
    print(f"\nTool {index}")
    print(f"Name: {tool.name}")
    print(f"Description: {tool.description}")

Loaded 14 tools from MCP server(s)

Available tools:

Tool 1
Name: read_file
Description: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.

Tool 2
Name: read_text_file
Description: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as text regardless of extension. Only works within allowed directories.

Tool 3
Name: read_media_file
Description: Read a file and return it as a base64-encoded content block with its MIME type. Image and audio files are returned as image/audio content; any other file type is returned as an embedded resource. Only works within allowed directories.

Tool 4
Name: read_multiple_files
Description:

In [24]:
import json

for tool in mcp_tools:
    print(f"\n--- {tool.name} ---")

    schema = getattr(tool, "args_schema", None)

    if schema is None:
        print("No input schema available.")

    elif isinstance(schema, dict):
        # Some versions return the schema directly as a dictionary
        print(json.dumps(schema, indent=2))

    elif hasattr(schema, "model_json_schema"):
        # Pydantic v2
        print(json.dumps(schema.model_json_schema(), indent=2))

    elif hasattr(schema, "schema"):
        # Pydantic v1
        print(json.dumps(schema.schema(), indent=2))

    else:
        print(repr(schema))


--- read_file ---
{
  "type": "object",
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "properties": {
    "path": {
      "type": "string"
    },
    "tail": {
      "description": "If provided, returns only the last N lines of the file",
      "type": "number"
    },
    "head": {
      "description": "If provided, returns only the first N lines of the file",
      "type": "number"
    }
  },
  "required": [
    "path"
  ]
}

--- read_text_file ---
{
  "type": "object",
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "properties": {
    "path": {
      "type": "string"
    },
    "tail": {
      "description": "If provided, returns only the last N lines of the file",
      "type": "number"
    },
    "head": {
      "description": "If provided, returns only the first N lines of the file",
      "type": "number"
    }
  },
  "required": [
    "path"
  ]
}

--- read_media_file ---
{
  "type": "object",
  "$schema": "https://json-schema.org/draft/2020-12/

In [25]:
for tool in mcp_tools:
    print(tool.name, type(tool.args_schema))

read_file <class 'dict'>
read_text_file <class 'dict'>
read_media_file <class 'dict'>
read_multiple_files <class 'dict'>
write_file <class 'dict'>
edit_file <class 'dict'>
create_directory <class 'dict'>
list_directory <class 'dict'>
list_directory_with_sizes <class 'dict'>
directory_tree <class 'dict'>
move_file <class 'dict'>
search_files <class 'dict'>
get_file_info <class 'dict'>
list_allowed_directories <class 'dict'>


In [26]:
for tool in mcp_tools:
    schema = tool.args_schema

    if isinstance(schema, dict):
        properties = schema.get("properties", {})
        required = schema.get("required", [])

        print(f"\nTool: {tool.name}")
        print("Arguments:", list(properties.keys()))
        print("Required:", required)


Tool: read_file
Arguments: ['path', 'tail', 'head']
Required: ['path']

Tool: read_text_file
Arguments: ['path', 'tail', 'head']
Required: ['path']

Tool: read_media_file
Arguments: ['path']
Required: ['path']

Tool: read_multiple_files
Arguments: ['paths']
Required: ['paths']

Tool: write_file
Arguments: ['path', 'content']
Required: ['path', 'content']

Tool: edit_file
Arguments: ['path', 'edits', 'dryRun']
Required: ['path', 'edits']

Tool: create_directory
Arguments: ['path']
Required: ['path']

Tool: list_directory
Arguments: ['path']
Required: ['path']

Tool: list_directory_with_sizes
Arguments: ['path', 'sortBy']
Required: ['path']

Tool: directory_tree
Arguments: ['path', 'excludePatterns']
Required: ['path']

Tool: move_file
Arguments: ['source', 'destination']
Required: ['source', 'destination']

Tool: search_files
Arguments: ['path', 'pattern', 'excludePatterns']
Required: ['path', 'pattern']

Tool: get_file_info
Arguments: ['path']
Required: ['path']

Tool: list_allowed_di

### Step 4: Create Agent with MCP Tools

In [33]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

tools = await mcp_client.get_tools()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant that uses the available filesystem "
        "tools when needed. Base file-related answers only on tool results."
    ),
)

print(f"Agent created with {len(tools)} MCP tools")
print(f"Tools: {[tool.name for tool in tools]}")

response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Use the available filesystem tools to list the files "
                    f"in this directory: {DOCUMENTS_DIR}"
                )
            )
        ]
    }
)

print(response["messages"][-1].content)

Agent created with 14 MCP tools
Tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']
The directory contains the following file:

- **eu_ai_act.pdf**


In [ ]:
# Testing agent
question = "What is the EU AI Act?"
print(f"Question: {question}\n")

result = await agent.ainvoke({
    "messages": [HumanMessage(content=question)]
})

print(f"Answer: {result['messages'][-1].content}")

Question: What is the EU AI Act?

Answer: The EU AI Act, formally known as the Artificial Intelligence Act, is a legislative proposal by the European Commission aimed at regulating artificial intelligence (AI) technologies within the European Union. It was introduced in April 2021 and is part of the EU's broader strategy to promote trustworthy AI while ensuring safety and fundamental rights.

Key aspects of the EU AI Act include:

1. **Risk-Based Classification**: The Act categorizes AI systems into four risk levels: unacceptable risk, high risk, limited risk, and minimal risk. Unacceptable risk AI systems are banned, while high-risk systems are subject to strict regulations.

2. **Requirements for High-Risk AI**: High-risk AI systems must comply with specific requirements related to data quality, documentation, transparency, human oversight, and robustness. This includes conducting risk assessments and ensuring that AI systems are designed to minimize risks.

3. **Transparency Obligat

### Step 5: Access MCP Resources

In [35]:
resources = []

async with mcp_client.session("filesystem") as session:
    try:
        resource_result = await session.list_resources()
        resources = resource_result.resources

        print(f"Found {len(resources)} resource(s).")

        for resource in resources:
            print(f"\nName: {resource.name}")
            print(f"URI: {resource.uri}")
            print(f"Description: {resource.description}")

    except Exception as error:
        print("This MCP server does not expose resources.")
        print(f"Details: {error}")

This MCP server does not expose resources.
Details: Method not found


- The configured filesystem MCP server exposed callable tools but did not expose MCP resources in this environment. The resource-listing attempt was handled without crashing, and filesystem tools remained available for the agent.

### Step 6: Build a Complete MCP-Enabled Agent

In [ ]:
# Reuse the mcp_client created in Step 2 and load tools

tools = await mcp_client.get_tools()

print(f"Loaded {len(tools)} MCP tools:")
for tool in tools:
    print(f"- {tool.name}")

Loaded 14 MCP tools:
- read_file
- read_text_file
- read_media_file
- read_multiple_files
- write_file
- edit_file
- create_directory
- list_directory
- list_directory_with_sizes
- directory_tree
- move_file
- search_files
- get_file_info
- list_allowed_directories


In [ ]:
# Create agent
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are an MCP-enabled assistant. "
        "Use the available MCP tools when the user's question requires "
        "filesystem information. "
        "Base factual claims about files only on tool results. "
        "If a tool fails or does not provide enough evidence, explain "
        "the limitation instead of guessing."
    ),
)

print("MCP-enabled agent created.")

MCP-enabled agent created.


In [76]:
# Testing agent
from langchain_core.messages import HumanMessage

test_query = (
    "Use the available filesystem MCP tool to list the files "
    f"in the MCP folder"
)

result = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(content=test_query)
        ]
    }
)

print("Agent run completed.")

Agent run completed.


In [ ]:
# Checking whether the agent produced tool calls and what evidence came back
import re

# Function to hide my folder path
def redact_output(value):
    # Convert lists, dictionaries, and message objects to text
    text = json.dumps(value, ensure_ascii=False, default=str)

    # Possible versions of the private path
    paths_to_hide = {
        str(DOCUMENTS_DIR),
        os.path.abspath(str(DOCUMENTS_DIR)),
        os.path.realpath(str(DOCUMENTS_DIR)),
        str(DOCUMENTS_DIR).replace("/", "\\"),
    }

    for path in paths_to_hide:
        if path:
            text = text.replace(path, "<MCP_FOLDER>")

    # Hide Unix-style absolute paths
    text = re.sub(
        r'(?<!https:)(?<!http:)(?<!\w)/(?:[^/\s"\\]+/)+[^/\s"\\]+',
        "<PRIVATE_PATH>",
        text,
    )

    # Hide Windows-style absolute paths
    text = re.sub(
        r'[A-Za-z]:\\\\(?:[^\\\\"]+\\\\)*[^\\\\"]+',
        "<PRIVATE_PATH>",
        text,
    )

    return json.loads(text) if text.startswith(("{", "[")) else text

# Printing the calls and evidence
for message in result["messages"]:
    print(f"\n[{message.type}]")
    print(redact_output(message.content))


[human]
"Use the available filesystem MCP tool to list the files in the MCP folder"

[ai]
""

[tool]
[{'type': 'text', 'text': 'Allowed directories:\n<PRIVATE_PATH>', 'id': 'lc_9cbff048-0948-4264-a2b4-9196c20bc18a'}]

[ai]
""

[tool]
[{'type': 'text', 'text': '[FILE] eu_ai_act.pdf', 'id': 'lc_eabcfc3b-921d-4a94-a7b6-9dee8bae5f95'}]

[ai]
"The MCP folder contains the following file:\n\n- **eu_ai_act.pdf**"


In [78]:
step_6_evidence = {
    "tool_count": len(tools),
    "tool_names": [tool.name for tool in tools],
    "query": test_query,
    "final_output": final_message.content,
}

step_6_evidence

{'tool_count': 14,
 'tool_names': ['read_file',
  'read_text_file',
  'read_media_file',
  'read_multiple_files',
  'write_file',
  'edit_file',
  'create_directory',
  'list_directory',
  'list_directory_with_sizes',
  'directory_tree',
  'move_file',
  'search_files',
  'get_file_info',
  'list_allowed_directories'],
 'query': 'Use the available filesystem MCP tool to list the files in the MCP folder',
 'final_output': 'The directory contains the following file:\n\n- **eu_ai_act.pdf**'}